# Replace the network
Change the `make_network` cell, then run the remaining cells. Your model receives normalized float32 features `[batch, 15]` and returns four finite scores `[batch, 4]`. Normalization is fitted only on training rows. No custom base class is needed.

In [ ]:
from pzr import student
import pandas as pd
import torch

torch.set_num_threads(1)
task = student.load_task("robot_arm", bundle="starter", budget=40)

In [ ]:
from torch import nn

def make_network(input_dim, num_actions):
    return nn.Sequential(
        nn.Linear(input_dim, 64), nn.ReLU(),
        nn.Linear(64, 32), nn.ReLU(),
        nn.Linear(32, num_actions),
    )

In [ ]:
config = student.TrainingConfig(epochs=20, seed=42)
baseline = student.train_selector(task, config=config)
challenger = student.train_selector(task, model_factory=make_network, config=config)
pd.DataFrame({
    "baseline": student.evaluate_selector(baseline.policy, task.validation),
    "my network": student.evaluate_selector(challenger.policy, task.validation),
})

## Inspect predictions
Costs are training/evaluation labels from a two-event terminal-loss teacher. They are never inputs to the policy. The `feasible` mask accompanies NaN costs for unavailable actions. Scores are rankings, not estimated physical costs.

In [ ]:
sample = 0
pd.DataFrame({"action": task.dataset.candidate_names,
              "score": challenger.policy.predict_scores(task.validation.features[sample]),
              "teacher_cost": task.validation.teacher_costs[sample],
              "feasible": task.validation.feasible[sample]})

## Compare native trajectories
Each method sees the same recorded validation inputs. These are prefix diagnostics; use full traces for scientific comparison.

In [ ]:
methods = {"baseline": baseline.policy, "challenger": challenger.policy,
           **{name: name for name in task.dataset.candidate_names}}
reports = {name: student.evaluate_monitor(policy, task.validation_traces, max_events=20,
                                          reference_cache="workspace/references")
           for name, policy in methods.items()}
pd.concat(reports, names=["method", "row"])[["final_approx_loss", "sum_approx_loss", "fallback_count"]]

## Custom training loops
The same prepared samples and pairwise loss are public. Keep validation out of gradient updates and normalization fitting.

In [ ]:
samples = student.prepared_samples(task.training, challenger.policy.normalizer)
features, teacher_costs, feasible = next(iter(torch.utils.data.DataLoader(samples, batch_size=32)))
scores = challenger.policy.model(features)
loss = student.cost_sensitive_pairwise_loss(scores.double(), teacher_costs, feasible)
print("Example batch loss:", float(loss.detach()))

In [ ]:
from pathlib import Path
from uuid import uuid4
output = Path("workspace") / ("custom-selector-" + uuid4().hex[:8])
challenger.save(output)
loaded = student.load_selector(output, model_factory=make_network)
print(student.evaluate_selector(loaded, task.validation))